# 🤖 CS Algorithm Q&A Assistant
## Step 1: Environment Setup & Base Model Test

**Model:** Google Gemma 2B  
**Goal:** Verify GPU is available, install dependencies, load Gemma 2B, and run test prompts.

---
⚠️ **Before running:** Go to `Runtime → Change runtime type → T4 GPU`

## ✅ Cell 1: Check GPU

In [ ]:
import subprocess

# Check GPU
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

# Check available VRAM
import os
gpu_info = result.stdout
if 'T4' in gpu_info:
    print('✅ T4 GPU detected — perfect for Gemma 2B!')
elif 'failed' in gpu_info.lower() or gpu_info == '':
    print('❌ No GPU found. Go to Runtime → Change runtime type → GPU (T4)')
else:
    print('✅ GPU detected:', gpu_info.split('\n')[8].strip() if len(gpu_info.split('\n')) > 8 else 'Unknown GPU')

## 📦 Cell 2: Install Dependencies

In [ ]:
# Install all required packages
!pip install -q transformers==4.38.2
!pip install -q accelerate==0.26.0
!pip install -q bitsandbytes==0.43.0
!pip install -q peft==0.9.0
!pip install -q datasets==2.18.0
!pip install -q trl==0.8.1
!pip install -q huggingface-hub

print('\n✅ All packages installed successfully!')

## 🔑 Cell 3: Hugging Face Login

Gemma requires accepting the license on HuggingFace first.

**Steps:**
1. Go to https://huggingface.co/google/gemma-2b
2. Accept the license agreement
3. Go to https://huggingface.co/settings/tokens
4. Create a token with **Read** access
5. Paste it below when prompted

In [ ]:
from huggingface_hub import login

# This will prompt you to enter your HuggingFace token
login()

print('✅ Logged in to Hugging Face!')

## 🧠 Cell 4: Verify Torch & CUDA

In [ ]:
import torch

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available:  {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'GPU Name:        {torch.cuda.get_device_name(0)}')
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'Total VRAM:      {total_vram:.1f} GB')
    print(f'\n✅ Ready for model loading!')
else:
    print('❌ CUDA not available — switch runtime to GPU')

## 📥 Cell 5: Load Gemma 2B Base Model

We load in **4-bit quantization** to save memory — this is the same setting we'll use during fine-tuning.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

MODEL_NAME = "google/gemma-2b"

print(f'Loading model: {MODEL_NAME}')
print('This may take 2-4 minutes on first run (downloading ~5GB)...\n')

# 4-bit quantization config — same as QLoRA training setup
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print('✅ Tokenizer loaded')

# Load model with 4-bit quantization
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

print('✅ Model loaded successfully!')

# Check memory usage
vram_used = torch.cuda.memory_allocated() / 1e9
vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'\n📊 VRAM used:  {vram_used:.2f} GB')
print(f'📊 VRAM total: {vram_total:.1f} GB')
print(f'📊 VRAM free:  {vram_total - vram_used:.2f} GB')

## 📊 Cell 6: Inspect Model Architecture

In [ ]:
# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print('=' * 50)
print('MODEL SUMMARY')
print('=' * 50)
print(f'Model:            {MODEL_NAME}')
print(f'Total parameters: {total_params / 1e9:.2f}B')
print(f'Trainable params: {trainable_params / 1e6:.1f}M')
print(f'Architecture:     {model.config.model_type}')
print(f'Hidden size:      {model.config.hidden_size}')
print(f'Num layers:       {model.config.num_hidden_layers}')
print(f'Num heads:        {model.config.num_attention_heads}')
print(f'Vocab size:       {model.config.vocab_size:,}')
print('=' * 50)

## 🧪 Cell 7: Helper Function for Generation

In [ ]:
def generate_response(instruction, max_new_tokens=300, temperature=0.7):
    """
    Generate a response from the base model given an instruction.
    Uses Gemma's chat format.
    """
    # Gemma prompt format
    prompt = f"<start_of_turn>user\n{instruction}<end_of_turn>\n<start_of_turn>model\n"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_length = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode only the new tokens (not the prompt)
    response = tokenizer.decode(
        outputs[0][input_length:],
        skip_special_tokens=True
    )
    return response.strip()


def print_response(instruction):
    print('=' * 60)
    print(f'📌 QUESTION:\n{instruction}')
    print('-' * 60)
    print('🤖 BASE MODEL RESPONSE:')
    response = generate_response(instruction)
    print(response)
    print('=' * 60)
    print()

print('✅ Helper functions ready!')

## 🧪 Cell 8: Test Prompt 1 — Binary Search

In [ ]:
print_response(
    "Explain binary search in Python with code. Include time and space complexity."
)

## 🧪 Cell 9: Test Prompt 2 — Two Sum

In [ ]:
print_response(
    "Solve the Two Sum problem in Python. Given an array of integers and a target, return indices of two numbers that add up to the target. Explain your approach and give the time complexity."
)

## 🧪 Cell 10: Test Prompt 3 — Big O

In [ ]:
print_response(
    "What is Big O notation? Explain O(1), O(log n), O(n), O(n log n), and O(n^2) with simple Python examples."
)

## 🧪 Cell 11: Test Prompt 4 — Recursion

In [ ]:
print_response(
    "Write a recursive Python function to compute the nth Fibonacci number. Then show an optimized version using memoization and compare their time complexities."
)

## 📊 Cell 12: Tokenizer Stats

In [ ]:
# Check how our test prompts tokenize — useful for planning dataset token lengths
test_prompts = [
    "Explain binary search in Python with code. Include time and space complexity.",
    "Solve the Two Sum problem in Python with a hashmap approach.",
    "What is the difference between BFS and DFS? When would you use each?",
    "Explain dynamic programming with the coin change problem in Python."
]

print('TOKEN LENGTH ANALYSIS')
print('-' * 50)
for prompt in test_prompts:
    tokens = tokenizer(prompt, return_tensors='pt')
    length = tokens['input_ids'].shape[1]
    print(f'{length:3d} tokens | {prompt[:55]}...')

print('\n💡 Target max_seq_length for training: 512 tokens')
print('   (covers instruction + detailed response with code)')

## ✅ Cell 13: Step 1 Summary

In [ ]:
import torch

vram_used = torch.cuda.memory_allocated() / 1e9
vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9

print('=' * 60)
print('🎉 STEP 1 COMPLETE — SUMMARY')
print('=' * 60)
print(f'✅ GPU:           {torch.cuda.get_device_name(0)}')
print(f'✅ Model:         google/gemma-2b loaded in 4-bit')
print(f'✅ VRAM used:     {vram_used:.2f} GB / {vram_total:.1f} GB total')
print(f'✅ VRAM headroom: {vram_total - vram_used:.2f} GB free for training')
print(f'✅ Base model:    generating responses (pre fine-tuning)')
print()
print('📋 OBSERVATIONS TO NOTE:')
print('   - Does the base model explain algorithms clearly?')
print('   - Does it include working code?')
print('   - Does it mention time/space complexity unprompted?')
print('   - Is the format consistent?')
print()
print('These observations will show WHY fine-tuning is needed.')
print()
print('📌 NEXT: Step 2 — Dataset Collection & Cleaning')
print('=' * 60)